Initial Analysis of the PN unit register data 

In [ ]:
import datetime as dt
from irish_electricity_data import MarketClient
from transform.general import pydantic_to_df
from storage.general import load_parquet



In [ ]:
with MarketClient() as client:
    data = client.semo.get_fpns(
        start_date=dt.date(2026, 10, 1),
        end_date=dt.date(2026, 10, 30),
    )

pn_df = pydantic_to_df(data)

pn_df

In [ ]:
number_of_units = pn_df['resource_name'].drop_duplicates().count()

print(number_of_units)

pn_df.describe()


In [ ]:
pn_ramp_df = pn_df[pn_df['start_mw'] != pn_df['end_mw']]
pn_ramp_df

The above periods look like ramp periods, where the units scale up to their dis/charging volume

Questions I want answered about the PN dataset:
- Find out if every charging and discharging period has a ramp period, and how to gracefully add this data to the non ramp charging periods to get the overall power
- Find out if the PN dataset is the best for analysing the charging and discharging data, or if I would be better with outturn generation data (does this cover charging as well as discharging?)
- Check if there is a 1-1 map between the unit data register from SEMO, the GU_IDs in the PN dataset with negative PNs (indicating charging) and the GU_IDs in the PN dataset with ramp periods. In my thinking at the moment, these should all be the same set of GU_IDs

In [ ]:
pn_charging_df = pn_df[(pn_df['start_mw'] < 0) | (pn_df['end_mw'] < 0 )]
pn_charging_df

In [ ]:
bess_unit_register_path = r'C:\Users\swift\Documents\Python Projects\SEM BESS Tracker\irish-bess-tracker\data\processed\BESS_Unit_Data_August_2026.parquet'

bess_unit_register_df = load_parquet(bess_unit_register_path)

In [ ]:
bess_unit_register_df